# Data Sourcing

In [48]:
import requests
import pandas as pd

In [49]:
## Functions
def summarize_data(df: pd.DataFrame):
    summary = {
    "head": df.head(),
    "info": df.info(),
    "describe": df.describe(include='all'),
    "missing": df.isnull().sum()
    }

    print(summary) 

## Sentiment Data

In [58]:

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey=demo'
r = requests.get(url)
data = r.json()

print(data)
# Read the JSON file into a pandas DataFrame
df_news = pd.DataFrame(data["feed"])

df_news = df_news.explode("ticker_sentiment")

ticker_df = pd.json_normalize(df_news["ticker_sentiment"])

df_news = df_news.drop(columns=["ticker_sentiment"])

df_news = pd.concat([df_news.reset_index(drop=True),
                ticker_df.reset_index(drop=True)], axis=1)


{'items': '50', 'sentiment_score_definition': 'x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish', 'relevance_score_definition': '0 < x <= 1, with a higher score indicating higher relevance.', 'feed': [{'title': 'ProShares S&P 500 High Income ETF Review and Aggregate Rating (ISPY)', 'url': 'https://www.marketbeat.com/stocks/BATS/ISPY/ratings/', 'time_published': '20260305T235236', 'authors': ['MarketBeat.com Staff'], 'summary': 'MarketBeat has assigned a "Moderate Buy" aggregate rating to the ProShares S&P 500 High Income ETF (ISPY), with an aggregate price target of $45.12. This rating is based on 821 analyst ratings over the past year, covering 25 companies, which constitute 42.7% of the ETF\'s portfolio. The report details the consensus ratings and price targets for ISPY\'s top 25 holdings, including major companies like NVIDIA, Apple, Microsoft, and Amazon.', 'banner_image': None, 'source': '

In [59]:
summarize_data(df_news)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   title                    114 non-null    object 
 1   url                      114 non-null    object 
 2   time_published           114 non-null    object 
 3   authors                  114 non-null    object 
 4   summary                  114 non-null    object 
 5   banner_image             101 non-null    object 
 6   source                   114 non-null    object 
 7   category_within_source   114 non-null    object 
 8   source_domain            114 non-null    object 
 9   topics                   114 non-null    object 
 10  overall_sentiment_score  114 non-null    float64
 11  overall_sentiment_label  114 non-null    object 
 12  ticker                   114 non-null    object 
 13  relevance_score          114 non-null    object 
 14  ticker_sentiment_score   1

In [52]:


# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol=IBM&apikey=demo'
r = requests.get(url)
data = r.json()

In [53]:
print(data)

# Read the JSON file into a pandas DataFrame
df = pd.DataFrame.from_dict(data, orient='index')

{'Meta Data': {'1. Information': 'Weekly Prices (open, high, low, close) and Volumes', '2. Symbol': 'IBM', '3. Last Refreshed': '2026-03-05', '4. Time Zone': 'US/Eastern'}, 'Weekly Time Series': {'2026-03-05': {'1. open': '235.7000', '2. high': '260.3800', '3. low': '233.7800', '4. close': '256.5500', '5. volume': '29165613'}, '2026-02-27': {'1. open': '254.3700', '2. high': '255.1900', '3. low': '220.7200', '4. close': '240.2100', '5. volume': '55457688'}, '2026-02-20': {'1. open': '259.2000', '2. high': '261.1100', '3. low': '253.5110', '4. close': '257.1600', '5. volume': '18536212'}, '2026-02-13': {'1. open': '295.9100', '2. high': '297.7200', '3. low': '256.6400', '4. close': '262.3800', '5. volume': '35501402'}, '2026-02-06': {'1. open': '307.5100', '2. high': '316.6400', '3. low': '278.9600', '4. close': '298.9300', '5. volume': '33862352'}, '2026-01-30': {'1. open': '293.1600', '2. high': '319.9000', '3. low': '291.2601', '4. close': '306.7000', '5. volume': '28527557'}, '2026-

In [54]:
# Extract Necessary Data
weekly = data["Weekly Time Series"]
meta_data = data["Meta Data"]

# seperate meta_data into DataFrame
meta_df = pd.DataFrame([meta_data])
# clean meta data columns
meta_df.columns = [c.split(". ")[1] for c in meta_df.columns]


# seperate weekly into DataFrame
df = pd.DataFrame.from_dict(weekly, orient="index")

df.columns = ["open","high","low","close","volume"]

df = df.astype(float)

df.index = pd.to_datetime(df.index)

df = df.sort_index()

print(meta_df[["Symbol","Last Refreshed"]])
print(df.head())

  Symbol Last Refreshed
0    IBM     2026-03-05
              open    high     low   close      volume
1999-11-12   90.50   97.50   90.50   95.87  43569700.0
1999-11-19   96.00  105.12   92.62  103.94  61550800.0
1999-11-26  105.50  109.87  101.81  105.00  37165600.0
1999-12-03  104.94  112.87  102.12  111.87  37670000.0
1999-12-10  113.00  122.12  107.56  109.00  58626000.0


In [55]:
# print(df.info())
# print(df.describe)

df.T

import pandas as pd

summary = {
    "info": df.info(),
    "describe": df.describe(include='all'),
    "missing": df.isnull().sum()
}

print(summary)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1374 entries, 1999-11-12 to 2026-03-05
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    1374 non-null   float64
 1   high    1374 non-null   float64
 2   low     1374 non-null   float64
 3   close   1374 non-null   float64
 4   volume  1374 non-null   float64
dtypes: float64(5)
memory usage: 64.4 KB
{'info': None, 'describe':               open         high          low        close        volume
count  1374.000000  1374.000000  1374.000000  1374.000000  1.374000e+03
mean    138.317323   141.592024   135.262330   138.493722  2.815655e+07
std      46.141788    47.128156    45.190578    46.235418  1.299055e+07
min      56.600000    62.000000    54.010000    56.600000  7.770697e+06
25%     104.000000   107.107500   100.625000   104.000000  1.858661e+07
50%     131.300000   134.900000   128.690000   131.270000  2.507098e+07
75%     162.392500   164.957500   159.927500 

In [56]:
def json_to_csv_pandas(df: pd.DataFrame, csv_file_path):
    
    # 2. Transpose dates into rows
    df.T
    
    # 3. Date name as Index
    df.index.name = 'Date'
    
    # Write the DataFrame to a CSV file
    df.to_csv(csv_file_path, index=False) # index=False prevents writing the DataFrame index to the CSV
    
    print(f"Conversion complete. CSV saved to {csv_file_path}")

In [57]:
# Example usage:
json_to_csv_pandas(data, 'test.csv')

AttributeError: 'dict' object has no attribute 'T'